In [77]:
import numpy as np
import plotly.subplots as sp
import plotly.graph_objects as go
from manim import *

# Constants

In [78]:
g = 9.81 # acceleration due to gravity on Earth (ms^-1)

# Euler's Method

In [79]:
def eulers_improved_method(state, step_size, derivative_function, *derivative_function_args):
    k1 = derivative_function(state, *derivative_function_args)
    trial = state + (step_size * k1)
    k2 = derivative_function(trial, *derivative_function_args)

    return (state + (step_size / 2 * (k1 + k2)))

# Derivatives

In [80]:
def derivatives(state, length):
    angle, angular_velocity = state

    return np.array([angular_velocity, ((-g / length) * np.sin(angle))])

# Pendulum Physics

In [ ]:
dt = 0.01 # step size
length = 1 # pendulum length
T = 2 * np.pi * np.sqrt(length / g) # time period of 1 oscillation
t_end = 10 * T # simulation runs for n time periods

t = np.arange(0, t_end + dt, dt) # each time step

## For Small Initial Angular Displacement

In [82]:
angles_small = np.zeros(len(t))
angular_velocities_small = np.zeros(len(t))

# initial conditions
angles_small[0] = np.radians(5)
angular_velocities_small[0] = 0

for i in np.arange(0, len(t) - 1, 1):
    state = eulers_improved_method(np.array([angles_small[i], angular_velocities_small[i]]), dt, derivatives, length)

    angles_small[i + 1] = state[0]
    angular_velocities_small[i + 1] = state[1]

## For Large Initial Angular Displacement

In [83]:
angles_large = np.zeros(len(t))
angular_velocities_large = np.zeros(len(t))

# initial conditions
angles_large[0] = np.radians(80)
angular_velocities_large[0] = 0

for i in np.arange(0, len(t) - 1, 1):
    state = eulers_improved_method(np.array([angles_large[i], angular_velocities_large[i]]), dt, derivatives, length)

    angles_large[i + 1] = state[0]
    angular_velocities_large[i + 1] = state[1]

# Plot of Angle vs Time

In [84]:
fig = sp.make_subplots(
    rows=1,
    cols=1, 
)

# Displacement 
fig.add_trace(go.Scatter(x=t, y=angles_small, name="Small Angle"), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=angles_large, name="Large Angle"), row=1, col=1)

fig.update_xaxes(title_text="Time / s", col=1)
fig.update_yaxes(title_text="Angle / rad", col=1, row=1)

fig.update_layout(
    title=f"Angle over time of a pendulum, length {length}m",
    template="plotly_dark",
    height=800
)

fig.show()

# Simulation

In [85]:
%%manim -qh -v WARNING SinglePendulumScene

scale = length * 3 # scale the pendulum for better visibility

# convert length and angle to cartesian coordinates
x = scale * np.sin(angles_large)
y = -scale * np.cos(angles_large)

class SinglePendulumScene(Scene):
    def construct(self):
        pivot = UP * 2

        pivot_point = np.array([0, 2, 0])

        bob = Dot(point=pivot_point + np.array([x[0], y[0], 0]), radius=0.1)

        string = Line(pivot_point, bob.get_center())

        tracker = ValueTracker(0)

        def get_bob_pos(frame):
            return pivot_point + np.array([x[frame], y[frame], 0])

        bob.add_updater(lambda b: b.move_to(
            get_bob_pos(int(tracker.get_value()))
        ))

        string.add_updater(lambda s: s.put_start_and_end_on(
            pivot_point,
            bob.get_center()
        ))

        self.add(bob, string)
        self.play(tracker.animate.set_value(len(x) - 1), run_time=t_end, rate_func=linear)


Manim Community v0.20.1

# Comments

When the pendulum is initially displaced by a large angle, the time period of that pendulum is longer than a pendulum initially displaced by a small angle.